In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, KFold
import fasttext
from lazypredict.Supervised import LazyClassifier
from transformers import BertTokenizer
import gensim
import gensim.downloader

dataset = pd.read_excel("Synthetic User Stories.xlsx")
dataset

,Domain Cluster,Topic,Domain,Machine Learning Task,User Story
0,Biology & Botanic,1,Biology,abstractive summarization,A group of researchers is using abstractive su...
1,Biology & Botanic,1,Plant Science,abstractive summarization,"As a plant scientist, I want to use abstractiv..."
2,Biology & Botanic,1,Biology,action model learning,"As a molecular biologist, I want to use action..."
3,Biology & Botanic,1,Plant Science,action model learning,"As a plant scientist, I want to use action mod..."
4,Biology & Botanic,1,Biology,activation function,"As a bioinformatics researcher, I want to use ..."
...,...,...,...,...,...
12396,Technical Domains,9,Computer Vision,word-sense disambiguation,"As a computer vision researcher, I want to use..."
12397,Technical Domains,9,Computer Networks,word2vec,"As a network engineer, I want to use word2vec ..."
12398,Technical Domains,9,Computer Vision,word2vec,"As a computer vision researcher, I want to use..."
12399,Technical Domains,9,Computer Networks,wordnet,"As a network engineer, I want to use WordNet t..."


In [2]:
target = []
for row in dataset.iterrows():
    target.append(np.where(dataset["Domain"].unique() == row[1]["Domain"])[0][0])
dataset["Target"] = target
dataset["Target"]

0         0
1         1
2         0
3         1
4         0
         ..
12396    37
12397    36
12398    37
12399    36
12400    37
Name: Target, Length: 12401, dtype: int64

In [3]:
def getTrainSetFastText():
    ft_model = fasttext.load_model("fasttext_model.bin")
    traindata = []
    for msg in dataset['User Story']:
        traindata.append(ft_model.get_sentence_vector(msg))
    traindata = pd.DataFrame(traindata)
    traindata.columns = traindata.columns.astype(str)
    return traindata

def getTrainSetTFIDF():
    countvec = CountVectorizer(max_features=100)
    bow = countvec.fit_transform(dataset['User Story']).toarray()
    tfidfconverter = TfidfTransformer()
    X = tfidfconverter.fit_transform(bow).toarray()
    training_data = pd.DataFrame(X)
    training_data.columns = training_data.columns.astype(str)
    return training_data

def getTrainSetBERT():
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    tokenized_data = tokenizer(dataset['User Story'].tolist(), padding=True, truncation=True, max_length=100)
    traindata = []
    for msg in tokenized_data['input_ids']:
        traindata.append(msg)
    traindata = pd.DataFrame(traindata)
    traindata.columns = traindata.columns.astype(str)
    return traindata

def getTrainSetWord2Vec():
    w2v_model = gensim.models.KeyedVectors.load_word2vec_format('word2vec-google-news-300.bin', binary=True)
    traindata = []
    for msg in dataset['User Story']:
        words = msg.split()
        vecs = []
        for word in words:
            if word in w2v_model:
                vecs.append(w2v_model[word][:100])
        if vecs:
            vec_avg = sum(vecs) / len(vecs)
        else:
            vec_avg = [0] * 100
        traindata.append(vec_avg)

    traindata = pd.DataFrame(traindata)
    traindata.columns = traindata.columns.astype(str)
    return traindata

def getTrainSetGlove():
    glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')
    traindata = []
    for msg in dataset['User Story']:
        words = msg.split()
        vecs = []
        for word in words:
            if word in glove_vectors:
                vecs.append(glove_vectors[word])
        if vecs:
            vec_avg = sum(vecs) / len(vecs)
        else:
            vec_avg = [0] * 100
        traindata.append(vec_avg)

    traindata = pd.DataFrame(traindata)
    traindata.columns = traindata.columns.astype(str)
    return traindata

In [ ]:
X = getTrainSetGlove() #Change this to get training set based on word embeddings method.
y = dataset['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=.5,random_state =123)
clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train, X_test, y_train, y_test)
models

In [ ]:
models[["Accuracy","F1 Score"]][:10]

In [4]:
result = pd.DataFrame(columns=["Fold","Model","Accuracy","F1-Score"], index=np.arange(300))
fold = KFold(n_splits=10, random_state=6666, shuffle=True)
X = getTrainSetWord2Vec() #Change this to get training set based on word embeddings method.
y = dataset['Target']
counter = 0
foldcounter = 1
for train_index, test_index in fold.split(X, y):
        print("Processing Fold "+ str(foldcounter) + " ...")
        X_train, X_test, y_train, y_test = \
            X[ X.index.isin(train_index)], X[ X.index.isin(test_index)], y[train_index], y[test_index]
        clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
        models,predictions = clf.fit(X_train, X_test, y_train, y_test)
        for model in models[:].iterrows():
            result.loc[counter]["Fold"] = foldcounter
            result.loc[counter]["Model"] = model[0]
            result.loc[counter]["Accuracy"] = round(model[1][0],3)
            result.loc[counter]["F1-Score"] = round(model[1][3],3)
            counter += 1
        foldcounter += 1
result

Processing Fold 1 ...


100%|██████████| 29/29 [03:42<00:00,  7.69s/it]


Processing Fold 2 ...


100%|██████████| 29/29 [03:42<00:00,  7.68s/it]


Processing Fold 3 ...


...

Processing Fold 9 ...


100%|██████████| 29/29 [03:50<00:00,  7.96s/it]


Processing Fold 10 ...


100%|██████████| 29/29 [03:53<00:00,  8.07s/it]


,Fold,Model,Accuracy,F1-Score
0,1,SVC,0.90,0.90
1,1,LogisticRegression,0.90,0.90
2,1,CalibratedClassifierCV,0.89,0.89
3,1,LinearSVC,0.89,0.89
4,1,LinearDiscriminantAnalysis,0.89,0.89
...,...,...,...,...
295,NaN,NaN,NaN,NaN
296,NaN,NaN,NaN,NaN
297,NaN,NaN,NaN,NaN
298,NaN,NaN,NaN,NaN


In [5]:
result = result.dropna()

In [6]:
result.to_excel("ResultWord2Vec_domain.xlsx")